# AquaSentinel AI — model training notebook

Use a Colab runtime with a GPU for the YOLOv8 fine-tune. This notebook uploads/extracts the project dataset from your Google Drive, trains a baseline model, evaluates on the held-out test split, then saves `best.pt` back to Drive. Keep the dataset's CC BY 4.0 attribution with the project.

Do not present metrics as project results until this run completes. Image detections are visual screening for **suspected** particles, not chemical identification.

In [ ]:
!pip -q install ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload Microplastic.v5i.yolov8.zip to this Drive location first.
DATASET_ZIP = '/content/drive/MyDrive/Microplastic.v5i.yolov8.zip'


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import yaml

extract_to = Path('/content/microplastic_dataset')
extract_to.mkdir(parents=True, exist_ok=True)
with ZipFile(DATASET_ZIP) as archive:
    archive.extractall(extract_to)

yaml_files = list(extract_to.rglob('data.yaml'))
if len(yaml_files) != 1:
    raise RuntimeError(f'Expected one data.yaml; found {len(yaml_files)}')
source_yaml = yaml_files[0]
dataset_root = source_yaml.parent
data = yaml.safe_load(source_yaml.read_text())
# The exported YAML uses ../train paths. Resolve each split against the actual extracted folder.
data['path'] = str(dataset_root)
for split, local_name in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
    candidate = dataset_root / local_name / 'images'
    if not candidate.exists():
        raise FileNotFoundError(f'Missing {split} images at {candidate}')
    data[split] = str(candidate)
normalized_yaml = Path('/content/aquasentinel-data.yaml')
normalized_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
print('Dataset root:', dataset_root)
print('Class labels:', data['names'])


In [ ]:
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Colab, choose Runtime > Change runtime type > T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
model = YOLO('yolov8n.pt')
train_result = model.train(
    data=str(normalized_yaml),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project='/content/aquasentinel-runs',
    name='microplastic-yolov8n',
    exist_ok=True,
    plots=True,
    seed=42,
)


In [ ]:
best_path = Path('/content/aquasentinel-runs/microplastic-yolov8n/weights/best.pt')
if not best_path.exists():
    raise FileNotFoundError(best_path)
best_model = YOLO(str(best_path))
test_metrics = best_model.val(data=str(normalized_yaml), split='test', imgsz=640, device=0, plots=True)
print(test_metrics.results_dict)


In [ ]:
from shutil import copy2
checkpoint_destination = Path('/content/drive/MyDrive/AquaSentinel-models/best.pt')
checkpoint_destination.parent.mkdir(parents=True, exist_ok=True)
copy2(best_path, checkpoint_destination)
print('Saved checkpoint to:', checkpoint_destination)


## After training

Download `best.pt` from `MyDrive/AquaSentinel-models/` and place it in the project's `weights/` folder. Set `MODEL_PATH=weights/best.pt`, then run the FastAPI service. Save the printed test metrics and training run details for the project report; do not copy sample metrics from any template.